# XGBoost

**Goal:** test whether a gradient boosting model improves on the Logistic Regression baseline (test F1 = 0.660, ROC-AUC = 0.860).

**How XGBoost works:** it builds hundreds of decision trees one after another. Each new tree focuses on the bookings the previous trees predicted wrongly and corrects part of their error (boosting). Unlike Logistic Regression, which adds up the effect of each feature separately, trees can learn interactions between features, e.g. "long lead time **and** non-refundable deposit **and** group booking".

**Data and evaluation:** the same preprocessed data, random split (70/15/15) and metrics as the baseline. Settings are chosen on the validation set; the test set is used once, for the final model.

In [3]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent                                  
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.data_split import main as create_data_splits, split_features_target      
from src.evaluation import evaluate

In [4]:
splits = create_data_splits()

X_train, y_train = split_features_target(splits["random"]["train"])         
X_val, y_val = split_features_target(splits["random"]["validation"])
X_test, y_test = split_features_target(splits["random"]["test"])

print(X_train.shape, X_val.shape, X_test.shape)

(59576, 29) (12766, 29) (12767, 29)


In [5]:
numerical_columns = X_train.select_dtypes(include="number").columns.tolist()   
categorical_columns = X_train.select_dtypes(exclude="number").columns.tolist()  

print(len(numerical_columns), "numerical:", numerical_columns)
print(len(categorical_columns), "categorical:", categorical_columns)

16 numerical: ['lead_time', 'stays_in_weekend_nights', 'stays_in_week_nights', 'adults', 'children', 'babies', 'is_repeated_guest', 'previous_cancellations', 'previous_bookings_not_canceled', 'booking_changes', 'days_in_waiting_list', 'adr', 'required_car_parking_spaces', 'total_of_special_requests', 'total_nights', 'total_guests']
13 categorical: ['hotel', 'arrival_date_month', 'meal', 'country', 'market_segment', 'distribution_channel', 'reserved_room_type', 'assigned_room_type', 'deposit_type', 'agent', 'company', 'customer_type', 'arrival_weekday']


## Encoding

- **Numerical features:** passed through unchanged (`"passthrough"`). Trees split on thresholds such as "lead_time > 90", so the scale of a feature does not matter and no scaling is needed.
- **Categorical features:** one-hot encoded as in the baseline, with categories that appear fewer than 50 times in the training set grouped as "infrequent".

In [6]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

preprocessor = ColumnTransformer([
    ("num", "passthrough", numerical_columns),                                  
    ("cat", OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=50), categorical_columns),
])

In [7]:
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

xgb = Pipeline([
    ("preprocessor", preprocessor),                                      
    ("model", XGBClassifier(random_state=42, n_jobs=-1)),      
])
xgb.fit(X_train, y_train)                                             

y_prob_xgb = xgb.predict_proba(X_val)[:, 1]                      
evaluate("XGBoost (default)", y_val , y_prob_xgb, "results_buse.csv", split="validation")   


XGBoost (default) (validation): F1=0.709, ROC-AUC=0.911


{'model': 'XGBoost (default)',
 'split': 'validation',
 'f1': 0.7085134933651409,
 'roc_auc': 0.9112654597419302,
 'precision': 0.7467001885606537,
 'recall': 0.6740425531914893}

**Interpretation:** with default settings, XGBoost already clearly outperforms Logistic Regression on the validation set (F1 0.709 vs 0.604, ROC-AUC 0.911 vs 0.862). Unlike class weighting in Logistic Regression, which only shifted the decision, XGBoost improves ROC-AUC, so it actually separates canceled and non-canceled bookings better. Precision (0.747) and recall (0.674) both improve compared with the default baseline. Recall is still limited: about a third of cancellations are missed, again because of the class imbalance.

## Hyperparameter tuning

We tried 12 combinations on the validation set (500 trees each):

| Parameter | Meaning | Values tried |
|---|---|---|
| `max_depth` | Maximum depth of each tree; deeper trees learn more complex rules but may overfit | 4, 6, 8 |
| `learning_rate` | How strongly each new tree corrects the previous ones | 0.05, 0.1 |
| `scale_pos_weight` | Extra weight for the canceled class; the balanced value is the ratio of non-canceled to canceled bookings (≈ 2.62) | 1, 2.62 |

In [8]:
from itertools import product

neg_pos_ratio = (y_train == 0).sum() / (y_train == 1).sum()        
print(f"scale_pos_weight for balanced classes: {neg_pos_ratio:.2f}")

for max_depth, learning_rate, spw in product([4, 6, 8], [0.05, 0.1], [1, neg_pos_ratio]):
    model = Pipeline([
        ("preprocessor", preprocessor),
        ("model", XGBClassifier(
            n_estimators=500,
            max_depth=max_depth,                                             
            learning_rate=learning_rate,
            scale_pos_weight=spw,
            random_state=42,
            n_jobs=-1,
        )),
    ])
    model.fit(X_train, y_train)
    y_prob = model.predict_proba(X_val)[:, 1]
    evaluate(f"XGBoost depth={max_depth} lr={learning_rate} spw={spw:.2f}",
             y_val, y_prob, "results_buse.csv", split="validation")

scale_pos_weight for balanced classes: 2.62
XGBoost depth=4 lr=0.05 spw=1.00 (validation): F1=0.690, ROC-AUC=0.905
XGBoost depth=4 lr=0.05 spw=2.62 (validation): F1=0.706, ROC-AUC=0.905
XGBoost depth=4 lr=0.1 spw=1.00 (validation): F1=0.705, ROC-AUC=0.909
XGBoost depth=4 lr=0.1 spw=2.62 (validation): F1=0.714, ROC-AUC=0.909
XGBoost depth=6 lr=0.05 spw=1.00 (validation): F1=0.702, ROC-AUC=0.910
XGBoost depth=6 lr=0.05 spw=2.62 (validation): F1=0.715, ROC-AUC=0.910
XGBoost depth=6 lr=0.1 spw=1.00 (validation): F1=0.709, ROC-AUC=0.912
XGBoost depth=6 lr=0.1 spw=2.62 (validation): F1=0.721, ROC-AUC=0.912
XGBoost depth=8 lr=0.05 spw=1.00 (validation): F1=0.706, ROC-AUC=0.912
XGBoost depth=8 lr=0.05 spw=2.62 (validation): F1=0.723, ROC-AUC=0.912
XGBoost depth=8 lr=0.1 spw=1.00 (validation): F1=0.713, ROC-AUC=0.913
XGBoost depth=8 lr=0.1 spw=2.62 (validation): F1=0.729, ROC-AUC=0.914


**Interpretation:**
- **`scale_pos_weight` raises F1 in every combination (by 0.01–0.02) but leaves ROC-AUC unchanged**, exactly like `class_weight` in Logistic Regression: weighting the minority class shifts the predicted probabilities and catches more cancellations, but does not change how the model ranks bookings.
- **Deeper trees perform better** (depth 4 → 6 → 8). The model benefits from more complex rules, and the validation scores show no sign of overfitting.
- **A learning rate of 0.1 is better than 0.05** with 500 trees; the smaller rate learns too slowly for this number of trees.

Best combination: `max_depth=8`, `learning_rate=0.1`, `scale_pos_weight≈2.62` (validation F1 = 0.729, ROC-AUC = 0.914). The gains between depth 6 and 8 are already small (ROC-AUC 0.912 → 0.914), so deeper trees are unlikely to help much.

In [10]:
best_xgb = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(
        n_estimators=500,
        max_depth=8,
        learning_rate=0.1,
        scale_pos_weight=2.62,
        random_state=42,
        n_jobs=-1,
    )),
])
best_xgb.fit(X_train, y_train)

y_prob_test = best_xgb.predict_proba(X_test)[:, 1]
evaluate("XGBoost", y_test, y_prob_test, "results_buse.csv", split="random_test")

XGBoost (random_test): F1=0.732, ROC-AUC=0.915


{'model': 'XGBoost',
 'split': 'random_test',
 'f1': 0.7321361295993972,
 'roc_auc': 0.9148015151305864,
 'precision': 0.6568273997296079,
 'recall': 0.826950354609929}

## Results

| Model | Set | F1 | ROC-AUC | Precision | Recall |
|---|---|---|---|---|---|
| XGBoost (default) | Validation | 0.709 | 0.911 | 0.747 | 0.674 |
| XGBoost (tuned) | Validation | 0.729 | 0.914 | – | – |
| **XGBoost (tuned)** | **Test** | **0.732** | **0.915** | **0.657** | **0.827** |

The test scores are almost identical to the validation scores, so the model generalizes well to unseen bookings.

**Comparison with the baseline (test set):**

| Model | F1 | ROC-AUC | Precision | Recall |
|---|---|---|---|---|
| Logistic Regression | 0.660 | 0.860 | 0.557 | 0.811 |
| XGBoost | 0.732 | 0.915 | 0.657 | 0.827 |

XGBoost is better on all four metrics. Both models catch about 81–83% of cancellations, but XGBoost does so with far fewer false alarms (precision 0.657 vs 0.557). Because it separates the two classes better, it pays a much smaller price in precision for a high recall.

**Next steps:** temporal test, ablation of the suspected leakage features, and feature importance (permutation importance and SHAP).